# Block 10: Tables, Figures, Appendix, and Reproducibility

**Goal**  
Inventory the generated artifacts and centralize a reproducibility manifest for the notebook suite.

**Checklist alignment**  
Wraps the outputs of Blocks 01-09 into a paper-facing inventory.

**Outputs**  
- tables under `results/tables/`
- figures under `results/figures/`
- manifests under `results/manifests/`
- executed notebook copy under `results/notebooks/` when this suite is run with `--execute`

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "experiment_checklist.md").exists() and (candidate / "implementation").exists():
            return candidate
    raise RuntimeError("Could not locate repo root for notebook execution.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from implementation.notebook_support import *

cfg = CanonicalConfig().validate()
dirs = ensure_results_dirs(cfg)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

In [ ]:
expected = pd.DataFrame(
    [
        ("block_01", "results/tables/block_01_claim_map.csv"),
        ("block_02", "results/manifests/block_02_split_manifest.json"),
        ("block_03", "results/tables/block_03_e12_real_summary.csv"),
        ("block_04", "results/tables/block_04_e4_crb.csv"),
        ("block_05", "results/tables/block_05_e2_bridge.csv"),
        ("block_06", "results/tables/block_06_e9_convergence.csv"),
        ("block_07", "results/tables/block_07_e6_ablation.csv"),
        ("block_08", "results/tables/block_08_pc_metrics.csv"),
        ("block_09", "results/tables/block_09_e10_nonstationary.csv"),
    ],
    columns=["block_id", "artifact_relpath"],
)
expected["exists"] = expected["artifact_relpath"].map(lambda p: (REPO_ROOT / p).exists())
display(expected)

In [ ]:
manifest_files = sorted(dirs["manifests"].glob("block_*.json"))
table_files = sorted(dirs["tables"].glob("block_*.*"))
figure_files = sorted(dirs["figures"].glob("block_*.*"))
inventory = pd.DataFrame(
    [{"kind": "manifest", "path": str(path.relative_to(REPO_ROOT))} for path in manifest_files]
    + [{"kind": "table", "path": str(path.relative_to(REPO_ROOT))} for path in table_files]
    + [{"kind": "figure", "path": str(path.relative_to(REPO_ROOT))} for path in figure_files]
)
display(inventory)

In [ ]:
manifest_path = dirs["manifests"] / "block_10_reproducibility_manifest.json"
save_json(
    {
        "canonical_config": config_as_frame(cfg).to_dict(orient="records"),
        "expected_artifacts": expected.to_dict(orient="records"),
        "present_inventory": inventory.to_dict(orient="records"),
    },
    manifest_path,
)
pd.DataFrame([manifest_row("block_10", "packaging", str(manifest_path.relative_to(REPO_ROOT)), cfg)])